# Week 9 Practical — Machine Learning Diagnostics, Model Selection & Precision–Recall
**Topics:** Debugging learning algorithms → train/CV/test → bias–variance → model selection → regularization → learning curves → error analysis → augmentation → transfer learning → fairness → skewed datasets → threshold tuning.

**Datasets:** `week9_housing.csv` and `week9_spam.csv` (synthetic teaching datasets; not real patient, housing, or email records).

**Student instructions:** Work sequentially. Every section contains a **TODO**. Replace the `# TODO` lines with executable code and answer the accompanying interpretation questions in a Markdown cell. Do not use the test set to choose features, hyperparameters, models, or thresholds. Submit your completed notebook with outputs and plots.

**Suggested duration:** two lab sessions or one extended assignment. **Packages:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`; TensorFlow is **optional** for the transfer-learning extension.

## Setup and reproducibility
Run this starter cell first. Put all three files in the same folder (or adjust `DATA_DIR`).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.metrics import (mean_squared_error, mean_absolute_error, accuracy_score,
    confusion_matrix, precision_score, recall_score, f1_score, precision_recall_curve,
    average_precision_score, classification_report)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
SEED = 42
DATA_DIR = Path('.')
housing = pd.read_csv(DATA_DIR / 'week9_housing.csv')
emails = pd.read_csv(DATA_DIR / 'week9_spam.csv')
print('Housing:', housing.shape, 'Email:', emails.shape)
display(housing.head(3), emails.head(3))

### TODO 0 — Inspect the datasets
1. Print column types, missing-value counts, and summary statistics for both datasets.
2. Plot housing prices against `size_sqft` and a bar chart of spam/non-spam frequencies.
3. Calculate the spam prevalence as a percentage. Explain why this distribution matters for accuracy.
4. Identify the target variable and at least three candidate predictors for each task.

In [ ]:
# TODO 0: inspect, visualize, and describe the two datasets.


## 1. Debugging a learning algorithm: establish a baseline
A baseline makes it possible to tell whether added model complexity helps. First use a single housing feature, then compare a richer model.

In [ ]:
Xh = housing.drop(columns='price_usd')
yh = housing['price_usd']
Xh_dev, Xh_test, yh_dev, yh_test = train_test_split(Xh, yh, test_size=.20, random_state=SEED)
Xh_train, Xh_val, yh_train, yh_val = train_test_split(Xh_dev, yh_dev, test_size=.25, random_state=SEED)
print(len(Xh_train), len(Xh_val), len(Xh_test))  # 60/20/20
baseline_reg = LinearRegression().fit(Xh_train[['size_sqft']], yh_train)
print('Baseline train RMSE:', np.sqrt(mean_squared_error(yh_train, baseline_reg.predict(Xh_train[['size_sqft']]))))
print('Baseline CV RMSE:', np.sqrt(mean_squared_error(yh_val, baseline_reg.predict(Xh_val[['size_sqft']]))))

### TODO 1 — Diagnose the baseline
1. Compute training and validation RMSE and MAE for the baseline.
2. Plot validation residuals (`true - predicted`) against predicted prices.
3. Identify any systematic patterns; explain whether they suggest underfitting, overfitting, or missing features.
4. Propose **two concrete next experiments**, each tied to an observed diagnostic rather than a guess.

In [ ]:
# TODO 1: implement RMSE/MAE, residual plot, and diagnostic notes.


## 2. Train, validation, and test sets
Training fits parameters; validation supports model/hyperparameter decisions; the held-out test set estimates final performance.

### TODO 2 — Check partitioning and leakage
1. Verify the 60%/20%/20% split sizes and ensure no row appears in multiple sets (use original indices).
2. Compare target distributions (mean, median, and standard deviation) across splits.
3. Explain why fitting a scaler or selecting a polynomial degree using all observations would leak information.
4. Explain why repeatedly choosing models based on test-set performance invalidates the final estimate.

In [ ]:
# TODO 2: verify disjoint indices and compare target distributions.


## 3. Bias, variance, and model complexity
Compare polynomial regressors of different degrees using the **same** train/validation split.

In [ ]:
degrees = [1, 2, 4, 8, 12]
# Each model uses only size_sqft, intentionally limiting available information.


### TODO 3 — Diagnose bias and variance
1. For each polynomial degree, build a `Pipeline([PolynomialFeatures(include_bias=False), StandardScaler(), Ridge(alpha=1.0)])` and fit **training data only**.
2. Record train and validation RMSE; plot both versus degree.
3. Identify a likely high-bias model and a likely high-variance model **based on the observed results**, not degree alone.
4. Explain what the training–validation gap tells you and why degree 12 is not automatically the best model.

In [ ]:
# TODO 3: fit models, store RMSEs, plot learning behavior.


## 4. Model selection and regularization (λ)
Ridge uses an L2 penalty; scikit-learn calls its regularization strength `alpha` (analogous to λ, with convention-dependent scaling).

In [ ]:
numeric_features = ['size_sqft','bedrooms','age_years','distance_km','quality_score']
categorical_features = ['neighborhood']
preprocess_h = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])
regularization_values = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]


### TODO 4 — Select λ without touching the test set
1. Build a preprocessing + `Ridge(alpha=value)` pipeline for each value above.
2. Fit on `Xh_train` and score on `Xh_val`; tabulate training and validation RMSE.
3. Plot validation RMSE against `alpha` on a logarithmic x-axis and select the lowest-validation-error value.
4. Refit your selected pipeline on the combined **training + validation** portion (`Xh_dev`, `yh_dev`).
5. Evaluate on the test set **once**, and report the final RMSE. State why the test set was not used during selection.

In [ ]:
# TODO 4: search alpha, select using validation, refit on dev, evaluate once on test.


## 5. Learning curves: more data or a better model?

### TODO 5 — Learning curves
1. Using only `Xh_dev` and `yh_dev`, construct learning curves for a simple linear model and a richer regularized model.
2. Use `learning_curve(..., cv=5, scoring="neg_root_mean_squared_error", train_sizes=np.linspace(.15, 1.0, 6))`.
3. Plot mean training and CV RMSE versus training sample size.
4. Explain which model would plausibly benefit more from additional data and which may need better features or capacity.
5. Do **not** use `Xh_test` in cross-validation.

In [ ]:
# TODO 5: generate learning curves and discuss bias/variance.


## 6. Iterative development and classification error analysis
We now use the email dataset to connect the debugging cycle to the spam example.

In [ ]:
emails = emails.copy()
emails['text'] = emails['subject'].fillna('') + ' ' + emails['body'].fillna('')
Xe_dev, Xe_test, ye_dev, ye_test = train_test_split(
    emails[['text','sender','error_category']], emails['is_spam'],
    test_size=.20, random_state=SEED, stratify=emails['is_spam'])
Xe_train, Xe_val, ye_train, ye_val = train_test_split(
    Xe_dev, ye_dev, test_size=.25, random_state=SEED, stratify=ye_dev)
text_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,1), min_df=2)),
    ('clf', LogisticRegression(max_iter=1000, class_weight=None, random_state=SEED))
])
text_model.fit(Xe_train['text'], ye_train)
val_scores = text_model.predict_proba(Xe_val['text'])[:,1]
val_preds = (val_scores >= .5).astype(int)
print(classification_report(ye_val, val_preds, zero_division=0))

### TODO 6 — Categorize actual mistakes
1. Identify validation emails for which the model prediction differs from the true label.
2. Display up to 15 misclassified emails with `subject`/`body` (recover them from the original dataset via index), true label, prediction, and `error_category`.
3. Count misclassified examples by category (e.g., phishing, pharma, finance, legitimate).
4. Suggest two targeted interventions: e.g., bigrams, sender-derived features, typo normalization, class weighting, or collecting more examples.
5. Explain why an error category with many examples may deserve attention even if it is not the easiest category to fix.

In [ ]:
# TODO 6: inspect validation errors and propose evidence-based improvements.


## 7. Data augmentation and feature engineering
**Important:** Apply augmentation only to training examples. Never augment validation/test examples or allow near-duplicates to leak across splits.

### TODO 7 — Controlled augmentation experiment
1. Implement a function that creates a typo-normalized text copy (e.g., replace `w4tches` → `watches`, `med1cine` → `medicine`) or a simple training-only word-variant augmentation.
2. Create a **training-only** augmented set; keep original labels unchanged.
3. Train a baseline and an augmented-data model with identical vectorizer/model settings.
4. Compare validation precision, recall, and F1 at threshold 0.5.
5. Discuss one risk of unrealistic augmentation or label corruption. Do not claim augmentation helped unless metrics support it.

In [ ]:
# TODO 7: create training-only augmented text, retrain, compare validation metrics.
def normalize_typos(text):
    # TODO: implement a simple, label-preserving normalization.
    return text


## 8. Transfer learning (conceptual experiment + optional TensorFlow)
Transfer learning reuses features/weights learned on a source task. A fair comparison keeps the target-data split and evaluation protocol fixed.

### TODO 8 — Design a transfer-learning comparison
1. Write a short experimental protocol for a pretrained text encoder (e.g., a TensorFlow Hub sentence encoder) versus TF-IDF + logistic regression.
2. Specify what is **frozen**, what is **trained**, and how you would prevent leakage.
3. Explain when pretrained representations might help with a small spam dataset and when source–target mismatch might hurt.
4. **Optional coding extension:** use a pretrained encoder available in your environment; compare validation F1 with the baseline. The core practical does not require an external model download.

In [ ]:
# TODO 8 (optional): code a pretrained encoder comparison if available.


## 9. Skewed datasets: why accuracy is misleading

In [ ]:
majority_baseline = DummyClassifier(strategy='most_frequent').fit(Xe_train[['text']], ye_train)
majority_pred = majority_baseline.predict(Xe_val[['text']])
print('Always-majority accuracy:', accuracy_score(ye_val, majority_pred))
print('Always-majority recall:', recall_score(ye_val, majority_pred, zero_division=0))

### TODO 9 — Evaluate class imbalance
1. Compute class counts and positive prevalence for train, validation, and test sets (do not use test scores yet).
2. Compute the confusion matrix, accuracy, precision, recall, and F1 for the majority baseline and logistic regression on **validation data**.
3. Explain how a classifier can achieve high accuracy but miss most spam messages.
4. Fit a second logistic regression model with `class_weight="balanced"`; compare validation metrics at 0.5.
5. Explain why resampling must be done **inside training folds**, not before splitting.

In [ ]:
# TODO 9: compare majority, standard logistic, and class-weighted logistic.


## 10. Trading off precision and recall with thresholds
For a binary classifier, predict spam if `P(spam|email) >= threshold`. Thresholds must be chosen on validation data.

In [ ]:
threshold_grid = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90, 0.99]
# val_scores and ye_val were defined in Section 6.


### TODO 10 — Choose an operating point
1. For each threshold in `threshold_grid`, compute validation precision, recall, F1, and the confusion matrix.
2. Plot precision and recall against threshold. Separately plot the precision–recall curve using `precision_recall_curve`.
3. Find the threshold maximizing validation F1. Then find a threshold that meets **recall ≥ 0.90**, if feasible, while maximizing precision.
4. Discuss which threshold is more appropriate for (a) a spam filter that must avoid discarding legitimate messages and (b) a screening system where missed positive cases are costly.
5. State why raising the threshold *typically* increases precision and lowers recall, but precision is **not guaranteed** to increase monotonically on finite datasets.

In [ ]:
# TODO 10: threshold table, plots, F1-optimal threshold, recall-constrained threshold.


## 11. Fairness, bias, and responsible evaluation
The synthetic dataset does **not** contain real demographic attributes; do not infer protected characteristics from names or addresses.

### TODO 11 — Audit performance across meaningful subgroups
1. Use an available non-sensitive grouping (e.g., `sender` domain or `error_category`) to compare false-positive and false-negative rates on the validation split. Note that this is a **subgroup reliability audit**, not a demographic fairness audit.
2. Explain what additional responsibly collected data and governance would be needed for a genuine fairness assessment.
3. Identify two potential harms from false positives and false negatives in spam filtering.
4. Propose a monitoring plan for post-deployment performance drift, error reporting, and periodic audits.

In [ ]:
# TODO 11: compute subgroup confusion matrices on validation data.


## 12. Final model, untouched test set, and reflection

### TODO 12 — Final evaluation and conclusions
1. Choose **one** text-classification pipeline and **one** threshold based only on previous validation experiments.
2. Refit the selected pipeline on **combined train + validation data** (`Xe_dev`, `ye_dev`); recompute its scores on the held-out `Xe_test`.
3. Apply the frozen threshold and report test accuracy, precision, recall, F1, confusion matrix, and average precision (AP).
4. Compare test and validation performance, but **do not** change the model or threshold after seeing test results.
5. Write a 200–300-word reflection: (i) strongest diagnostic, (ii) evidence for the chosen improvement, (iii) precision–recall trade-off, (iv) one limitation of synthetic data, (v) one deployment monitoring action.
6. Submit the completed `.ipynb` with plots and interpretations.

In [ ]:
# TODO 12: refit selected model on dev, evaluate once on test, and summarize.


## Student submission checklist
- [ ] All 13 TODO sections attempted with executable code and written interpretations.
- [ ] At least four informative plots (residuals, complexity, learning curve, PR/threshold).
- [ ] Training/validation/test discipline maintained throughout.
- [ ] A table comparing model choices and regularization strengths.
- [ ] Error-analysis table and justified next-step recommendations.
- [ ] Final test metrics reported **once** after model and threshold selection.

## References
- TensorFlow/Keras built-in datasets: https://keras.io/api/datasets/ (IMDB is available but is sentiment data, not spam).
- Scikit-learn precision–recall example: https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html
- Scikit-learn documentation: https://scikit-learn.org/stable/modules/model_evaluation.html

**Note:** Synthetic labels and categories are for classroom experimentation; no clinical or real-world fairness claims should be drawn from them.